# Training with RLlib PPO
[RLlib](https://docs.ray.io/en/latest/rllib/index.html) is a high-performance, distributed
reinforcement learning library. It is preferable to other RL libraries (e.g. Stable Baselines
3) for `bsk_rl` environments because it steps environments copies asynchronously; because 
of the variable step lengths, variable episode step counts, and long episode reset times, 
stepping each environment independently can increase step throughput by 2-5 times.

<div class="alert alert-warning">

**Warning:** RLlib had a bug that results in an undesirable timeout which stops
training. It has since been resolved: https://github.com/ray-project/ray/pull/45147

</div>

RLlib is actively developed and can change significantly from version to version. For this
script, the following version is used:

In [1]:
from importlib.metadata import version
from typing import ClassVar

version("ray")  # Parent package of RLlib

'2.35.0'

## Define the Environment
A nadir-scanning environment is created, to the one used in [this paper](https://hanspeterschaub.info/Papers/Stephenson2024.pdf). 
The satellite has to collect data while managing the data buffer level and battery level.

First, the satellite class is defined. A custom dynamics model is created that defines
a few additional properties to use in the state.

In [2]:
import numpy as np

from bsk_rl import act, data, obs, sats, scene
from bsk_rl.sim import dyn, fsw


class ScanningDownlinkDynModel(dyn.ContinuousImagingDynModel, dyn.GroundStationDynModel):
    # Define some custom properties to be accessed in the state
    @property
    def instrument_pointing_error(self) -> float:
        r_BN_P_unit = self.r_BN_P/np.linalg.norm(self.r_BN_P) 
        c_hat_P = self.satellite.fsw.c_hat_P
        return np.arccos(np.dot(-r_BN_P_unit, c_hat_P))
    
    @property
    def solar_pointing_error(self) -> float:
        a = self.world.gravFactory.spiceObject.planetStateOutMsgs[
            self.world.sun_index
        ].read().PositionVector
        a_hat_N = a / np.linalg.norm(a)
        nHat_B = self.satellite.sat_args["nHat_B"]
        NB = np.transpose(self.BN)
        nHat_N = NB @ nHat_B
        return np.arccos(np.dot(nHat_N, a_hat_N))

class ScanningSatellite(sats.AccessSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(
            dict(prop="storage_level_fraction"),
            dict(prop="battery_charge_fraction"),
            dict(prop="wheel_speeds_fraction"),
            dict(prop="instrument_pointing_error", norm=np.pi),
            dict(prop="solar_pointing_error", norm=np.pi)
        ),
        obs.OpportunityProperties(
            dict(prop="opportunity_open", norm=5700),
            dict(prop="opportunity_close", norm=5700),
            type="ground_station",
            n_ahead_observe=1,
        ),
        obs.Eclipse(norm=5700),
        obs.Time(),
    ]
    action_spec: ClassVar[list[act.Action]] = [
        act.Scan(duration=180.0),
        act.Charge(duration=120.0),
        act.Downlink(duration=60.0),
        act.Desat(duration=60.0),
    ]
    dyn_type = ScanningDownlinkDynModel
    fsw_type = fsw.ContinuousImagingFSWModel

Next, parameters are set. Since this scenario is focused on maintaining acceptable data
and power levels, these are tuned to create a sufficiently interesting mission.

In [3]:
sat = ScanningSatellite(
    "Scanner-1",
    sat_args=dict(
        # Data
        dataStorageCapacity=5000 * 8e6,  # bits
        storageInit=lambda: np.random.uniform(0.0, 0.8) * 5000 * 8e6,
        instrumentBaudRate=0.5 * 8e6,
        transmitterBaudRate=-50 * 8e6,
        # Power
        batteryStorageCapacity=200 * 3600,  # W*s
        storedCharge_Init=lambda: np.random.uniform(0.3, 1.0) * 200 * 3600,
        basePowerDraw=-10.0,  # W
        instrumentPowerDraw=-30.0,  # W
        transmitterPowerDraw=-25.0,  # W
        thrusterPowerDraw=-80.0,  # W
        panelArea=0.25,
        # Attitude
        imageAttErrorRequirement=0.1,
        imageRateErrorRequirement=0.1,
        disturbance_vector=lambda: np.random.normal(scale=0.0001, size=3),  # N*m
        maxWheelSpeed=6000.0,  # RPM
        wheelSpeeds=lambda: np.random.uniform(-3000, 3000, 3),
        desatAttitude="nadir",
    )
)

Finally, the environment arguments are set. Stepping through this environment is 
demonstrated at the bottom of the page.

In [4]:
duration = 5 * 5700.0  # About 5 orbits
env_args = dict(
    satellite=sat,
    scenario=scene.UniformNadirScanning(value_per_second=1/duration),
    rewarder=data.ScanningTimeReward(),
    time_limit=duration,
    failure_penalty=-1.0,
    terminate_on_time_limit=True,
)

## Set Up Custom Logging

The `bsk_rl` package supplies a utility to make logging information at the end of episodes
easier. This is useful to see how an agent's policy is changing over time, using a
monitoring program such as [TensorBoard](https://www.tensorflow.org/tensorboard). The callback 
is configured by writing a function that takes the environment as an input and returns a
dictionary with values to be logged.

In [5]:
def episode_data_callback(env):
    reward = env.rewarder.cum_reward
    reward = sum(reward.values()) / len(reward)
    orbits = env.simulator.sim_time / (95 * 60)

    data = dict(
        reward=reward,
        # Are satellites dying, and how and when?
        alive=float(env.satellite.is_alive()),
        rw_status_valid=float(env.satellite.dynamics.rw_speeds_valid()),
        battery_status_valid=float(env.satellite.dynamics.battery_valid()),
        orbits_complete=orbits,
    )
    if orbits > 0:
        data["reward_per_orbit"] = reward / orbits
    if not env.satellite.is_alive():
        data["orbits_complete_partial_only"] = orbits
        
    return data

## Configure Ray and PPO

PPO (or some other algorithm) can be configured. Of particular importance
are setting `sample_timeout_s` and `metrics_episode_collection_timeout_s` to appropriately
high values for this environment. The `episode_data_callback` is included in the environment
arguments, and the `WrappedEpisodeDataCallbacks` must be included in training to trigger
logging.

In [6]:
import bsk_rl.utils.rllib  # noqa To access "SatelliteTasking-RLlib"
from ray.rllib.algorithms.ppo import PPOConfig
from bsk_rl.utils.rllib.callbacks import WrappedEpisodeDataCallbacks

N_CPUS = 3

training_args = dict(
    lr=0.00003,
    gamma=0.999,
    train_batch_size=250,  # usually a larger number, like 2500
    num_sgd_iter=10,
    model=dict(fcnet_hiddens=[512, 512], vf_share_layers=False),
    lambda_=0.95,
    use_kl_loss=False,
    clip_param=0.1,
    grad_clip=0.5,
)

config = (
    PPOConfig()
    .training(**training_args)
    .env_runners(num_env_runners=N_CPUS-1, sample_timeout_s=1000.0)
    .environment(
        env="SatelliteTasking-RLlib",
        env_config=dict(**env_args, episode_data_callback=episode_data_callback),
    )
    .reporting(
        metrics_num_episodes_for_smoothing=1,
        metrics_episode_collection_timeout_s=180,
    )
    .checkpointing(export_native_model_files=True)
    .framework(framework="torch")
    .api_stack(
        enable_rl_module_and_learner=True,
        enable_env_runner_and_connector_v2=True,
    )
    .callbacks(WrappedEpisodeDataCallbacks)
)

Once the PPO configuration has been set, `ray` can be started and the agent can be
trained.

Training on a reasonably modern machine, we can achieve 5M steps over 32 processors in 6
to 18 hours, depending on specific environment configurations.

Note that the custom logging metrics are reported under env_runners.

In [7]:
import ray
from ray import tune

ray.init(
    ignore_reinit_error=True,
    num_cpus=N_CPUS,
    object_store_memory=2_000_000_000,  # 2 GB
)

# Run the training
tune.run(
    "PPO",
    config=config.to_dict(),
    stop={"training_iteration": 10},  # Adjust the number of iterations as needed
    checkpoint_freq=10,
    checkpoint_at_end=True
)

# Shutdown Ray
ray.shutdown()

2026-07-28 23:39:31,975	INFO worker.py:1783 -- Started a local Ray instance.


2026-07-28 23:39:34,683	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/gymnasium/spaces/box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:164: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:188: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


(PPO pid=4765) Install gputil for GPU system monitoring.
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,154 utils.orbital                  WARNING    <0.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,161 utils.orbital                  WARNING    <60.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,167 utils.orbital                  WARNING    <120.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,174 utils.orbital                  WARNING    <240.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,185 utils.orbital                  WARNING    <420.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,190 utils.orbital                  WARNING    <480.00> Could not

(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,200 utils.orbital                  WARNING    <660.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,208 utils.orbital                  WARNING    <780.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,214 utils.orbital                  WARNING    <840.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,219 utils.orbital                  WARNING    <900.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,224 utils.orbital                  WARNING    <960.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:47,229 utils.orbital                  WARNING    <1020.00> Could not find eclipse transitions in next 12000.0 seconds
(Si

...

(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,217 utils.orbital                  WARNING    <3720.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,225 utils.orbital                  WARNING    <3840.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,235 utils.orbital                  WARNING    <4020.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,243 utils.orbital                  WARNING    <4140.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,253 utils.orbital                  WARNING    <4320.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,260 utils.orbital                  WARNING    <4380.00> Could not find eclipse transitions in next 12000.0 second

(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,419 utils.orbital                  WARNING    <6840.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,429 utils.orbital                  WARNING    <7020.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,434 utils.orbital                  WARNING    <7080.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,441 utils.orbital                  WARNING    <7200.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,447 utils.orbital                  WARNING    <7260.00> Could not find eclipse transitions in next 12000.0 seconds
(SingleAgentEnvRunner pid=4814) 2026-07-28 23:39:58,452 utils.orbital                  WARNING    <7320.00> Could not find eclipse transitions in next 12000.0 second

(PPO pid=4765) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/runner/ray_results/PPO_2026-07-28_23-39-34/PPO_SatelliteTasking-RLlib_96c42_00000_0_2026-07-28_23-39-34/checkpoint_000000)
2026-07-28 23:39:58,637	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/runner/ray_results/PPO_2026-07-28_23-39-34' in 0.0157s.


2026-07-28 23:39:58,989	INFO tune.py:1041 -- Total run time: 24.31 seconds (23.91 seconds for the tuning loop).


(SingleAgentEnvRunner pid=4815) 2026-07-28 23:39:57,379 sats.satellite.Scanner-1       WARNING    <27660.00> Scanner-1: failed battery_valid check


## Loading the Policy Network

The policy network can be found in the `p0` subdirectory of the checkpoint output, if 
using the torch backend, and the `model` subdirectory of the checkpoint output. Use
`bsk_rl.utils.rllib.load_torch_mlp_policy` to load torch policies.

## Stepping Through the Environment

The environment is stepped through with random actions to give a sense of how it acts.

In [8]:
from bsk_rl import SatelliteTasking

env = SatelliteTasking(**env_args, log_level="INFO")
env.reset()
terminated = False
while not terminated:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

2026-07-28 23:40:00,409 gym                            INFO       Resetting environment with seed=1112082691


2026-07-28 23:40:00,468 sats.satellite.Scanner-1       INFO       <0.00> Scanner-1: Finding opportunity windows from 0.00 to 28800.00 seconds


2026-07-28 23:40:00,506 gym                            INFO       <0.00> Environment reset


2026-07-28 23:40:00,506 gym                            INFO       <0.00> === STARTING STEP ===


2026-07-28 23:40:00,507 sats.satellite.Scanner-1       INFO       <0.00> Scanner-1: action_downlink tasked for 60.0 seconds


...

2026-07-28 23:40:02,759 sats.satellite.Scanner-1       INFO       <23700.00> Scanner-1: Satellite Scanner-1 requires retasking


2026-07-28 23:40:02,760 sats.satellite.Scanner-1       WARNING    <23700.00> Scanner-1: failed battery_valid check


2026-07-28 23:40:02,761 gym                            INFO       <23700.00> Step reward: -1.0


2026-07-28 23:40:02,761 gym                            INFO       <23700.00> Episode terminated: True


2026-07-28 23:40:02,761 gym                            INFO       <23700.00> Episode truncated: False
